<a href="https://colab.research.google.com/github/CodeTigs/Desafio_DIO_Treinamento_de_Redes_Neurais_com_Transfer_Learning/blob/main/Treinamento_de_Redes_Neurais_com_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importações e Download do Dataset**

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# Baixa e carrega o dataset 'cats_vs_dogs' já dividindo as fatias (splits)
(raw_train, raw_validation, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

print(f"Quantidade de imagens no dataset original: {metadata.splits['train'].num_examples}")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.GSUOXY_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Quantidade de imagens no dataset original: 23262


**Pré-processamento das Imagens**

In [2]:
IMG_SIZE = 160
BATCH_SIZE = 32

def format_example(image, label):
    # Redimensiona a imagem
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

# Aplica a formatação nos datasets
train = raw_train.map(format_example)
validation = raw_validation.map(format_example)
test = raw_test.map(format_example)

# Embaralha o treino e divide em lotes
train_batches = train.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
validation_batches = validation.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

**Criando o Modelo com MobileNetV2**

In [3]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# 1. Carrega a MobileNetV2 sem a camada de classificação final (include_top=False)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')

# 2. Congela a base para não destruir os pesos pré-treinados
base_model.trainable = False

# 3. Monta o modelo final com a nova camada de classificação
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = preprocess_input(inputs) # Aplica a normalização específica da MobileNetV2
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

# 4. Compila o modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

**Treinamento**

In [ ]:
epochs = 5 # 5 épocas são suficientes para um ótimo resultado com Transfer Learning

print("Iniciando o treinamento...")
history = model.fit(
    train_batches,
    epochs=epochs,
    validation_data=validation_batches
)

Iniciando o treinamento...
Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 367s 620ms/step - accuracy: 0.8648 - loss: 0.3181 - val_accuracy: 0.9630 - val_loss: 0.1348
Epoch 2/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 358s 613ms/step - accuracy: 0.9662 - loss: 0.1118 - val_accuracy: 0.9725 - val_loss: 0.0877
Epoch 3/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 0s 546ms/step - accuracy: 0.9751 - loss: 0.0846

**Avaliação dos Resultados**

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Acurácia (Treino)')
plt.plot(val_acc, label='Acurácia (Validação)')
plt.legend(loc='lower right')
plt.title('Evolução da Acurácia')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Erro (Treino)')
plt.plot(val_loss, label='Erro (Validação)')
plt.legend(loc='upper right')
plt.title('Evolução do Erro (Loss)')
plt.show()